# Ghostlock Vivo Y22 — Colab Build (CPU only)

Builds and analyzes on a plain Colab CPU runtime. **No GPU required.**

> The exploit does **not** run here. It runs on the physical Vivo Y22.
> Colab builds and analyzes; it cannot reach your phone over USB.

Runtime → Runtime type → **CPU only** (the default). Then Run all.

## 1. Clone

In [ ]:
!git clone --depth 1 https://github.com/cocyce459-oss/ghostlock-vivo-y22.git
%cd ghostlock-vivo-y22
!ls

## 2. Dependencies
Analysis needs `capstone` + `pyelftools`. UI extras are only for `src/app/main.py`.

In [ ]:
!pip install -q capstone pyelftools rich typer psutil gdown

## 3. Fetch vmlinux
Must be a real ELF — check the `file` output before going on. An HTML Drive error page will silently waste your time.

In [ ]:
!gdown "1yX1l9SKl-oFFru-prRG-UmCiLfC-Y8MN" -O Kernel.elf
!file Kernel.elf

## 4. Analyze
`-v` is **verbose**. The path flag is `--vmlinux` (short `-k`).

In [ ]:
!python3 tools/analyze_vmlinux.py --vmlinux Kernel.elf --device vivo-y22 --verbose

> **If the output says `KERNEL VERSION MISMATCH`, stop here.**
> The `target.h` offsets are not valid for that kernel build, and using
> them will very likely panic the device. Re-derive first — see
> `docs/OFFSETS.md`.

## 5. Host build (x86-64, for testing only)

In [ ]:
!cd src/core/exploit && make host && cd -

## 6. Android build (NDK)

In [ ]:
!wget -q https://dl.google.com/android/repository/android-ndk-r26d-linux.zip
!unzip -q android-ndk-r26d-linux.zip
!rm -f android-ndk-r26d-linux.zip  # Colab disk is finite

In [ ]:
import os
os.environ["ANDROID_NDK_HOME"] = "/content/android-ndk-r26d"
os.environ["PATH"] = (
    os.environ["ANDROID_NDK_HOME"]
    + "/toolchains/llvm/prebuilt/linux-x86_64/bin:"
    + os.environ["PATH"]
)
print("NDK toolchain on PATH")

In [ ]:
!cd src/core/exploit && PATH="/content/android-ndk-r26d/toolchains/llvm/prebuilt/linux-x86_64/bin:$PATH" make android && cd -
!file src/core/exploit/ghostlock_y22

> `file` **must** report `ARM aarch64`. If it reports x86-64, the build is
> misconfigured — do not push it to the device.

## 7. Read-only preflight

In [ ]:
!VMLINUX=Kernel.elf bash scripts/preflight.sh

## 8. Download artifacts

In [ ]:
from google.colab import files
files.download("src/core/exploit/ghostlock_y22")

## On your own machine
```bash
adb devices
adb push ghostlock_y22 /data/local/tmp/
adb shell chmod +x /data/local/tmp/ghostlock_y22
```
**Read `docs/safety-guide.md` first.** Then:
```bash
adb shell /data/local/tmp/ghostlock_y22
```